# OAI-iMorphics (contour .mat) → nnU-Net v2 converter

Pipeline hoàn chỉnh (đã hiệu chỉnh theo dữ liệu thật):
- Nhãn là **contour** trong `.mat` → **rasterize** thành mask.
- Ghép với ảnh **OAI DESS** (đã tải, đặt tên theo **barcode**) để lấy affine + kênh ảnh.
- Ảnh DESS là **slice-first `(160, 384, 384)`** → nhãn được transpose + xoay/đảo cho khớp:
  `transpose(2,0,1)` → `rot90 K=1` → `REV (đảo slice)`.

6 cấu trúc: Femoral / Med+Lat Tibial cartilage / Med+Lat Meniscus / Patellar. **Không có bone.**


In [ ]:
!pip install -q nibabel scipy scikit-image


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 0) Cấu hình


In [ ]:
from pathlib import Path

# --- Nguon iMorphics: thu muc DA GIAI NEN tren Drive (khong con dung zip) ---
IMORPH_ZIP = None
IMORPH_DIR = Path("/content/drive/MyDrive/iMorphics no DESS")

# --- Anh DESS da tai (dat ten theo barcode: 016610xxxxxx.nii.gz) ---
LABEL_ONLY = False
DESS_DIR   = Path("/content/drive/MyDrive/OAI_DESS")

# --- Output nnU-Net ---
DATASET_ID   = 12
DATASET_NAME = f"Dataset{DATASET_ID:03d}_iMorphics"
OUT_ROOT  = Path("/content/drive/MyDrive/nnUNet_raw") / DATASET_NAME
IMAGES_TR = OUT_ROOT / "imagesTr"
LABELS_TR = OUT_ROOT / "labelsTr"
for d in (IMAGES_TR, LABELS_TR):
    d.mkdir(parents=True, exist_ok=True)

CHANNEL_NAME = "MRI"
DEFAULT_SPACING = (0.7, 0.365, 0.365)   # (NS,H,W) OAI DESS, chi dung khi LABEL_ONLY
IMG_DEFAULT_HW  = 384                    # in-plane khi LABEL_ONLY

# --- Huong nhan de KHOP anh (da xac dinh qua QC) ---
LAB_ROT_K = 1        # rot90 in-plane (axes H,W)
LAB_FLIP  = False    # lat truc H
LAB_REV   = True     # dao thu tu slice

# contour .mat -> union label id (0 = bo)
STRUCTS = {
    "FemoralCartilage":2, "MedialTibialCartilage":4, "LateralTibialCartilage":5,
    "MedialMeniscus":6, "LateralMeniscus":7, "PatellarCartilage":8,
}
UNION_LABEL_NAMES = {
    "background":0, "femoral_cartilage":2, "medial_tibial_cartilage":4,
    "lateral_tibial_cartilage":5, "medial_meniscus":6, "lateral_meniscus":7,
    "patellar_cartilage":8,
}
print("LABEL_ONLY =", LABEL_ONLY, "| OUT_ROOT =", OUT_ROOT)


## 1) Liệt kê file `.mat`


In [ ]:
mat_paths = sorted(IMORPH_DIR.rglob("*.mat"))
print("Tong .mat:", len(mat_paths))
assert mat_paths, f"Khong thay .mat trong {IMORPH_DIR}"
print("Vi du:", mat_paths[0])


## 2) Hàm rasterize + build_label (khớp ảnh slice-first) + parse tên


In [ ]:
import numpy as np, shutil
from scipy.io import loadmat
from skimage.draw import polygon

def parse_case(mat_path):
    p = Path(mat_path)
    stem = p.stem
    toks = stem.split("_")
    subject, barcode = toks[0], toks[-1]
    visit = next((x for x in p.parts if x in ("V00","V01")), p.parent.name)
    side  = "L" if "LEFT" in stem else ("R" if "RIGHT" in stem else "X")
    return dict(case_id=f"{subject}_{visit}_{side}", subject=subject,
                visit=visit, side=side, barcode=barcode, stem=stem)

def _pieces(val):
    if val is None or np.size(val) == 0:
        return []
    a = np.asarray(val)
    if a.dtype == object:
        return [np.asarray(p, dtype=float) for p in val]
    arr = a.astype(float)
    return [arr] if (arr.ndim == 2 and arr.shape[1] == 3) else []

def rasterize_mat(mat_path, H, W):
    ds = loadmat(str(mat_path), squeeze_me=True, struct_as_record=False)["datastruct"]
    vol = np.zeros((H, W, len(ds)), dtype=np.uint8)
    for si in range(len(ds)):
        for name, lab in STRUCTS.items():
            if lab == 0: continue
            for pc in _pieces(getattr(ds[si], name, None)):
                if pc.shape[0] < 3: continue
                rr, cc = polygon(pc[:, 1], pc[:, 0], shape=(H, W))   # y=row, x=col
                vol[rr, cc, si] = lab
    return vol   # (H, W, NS)

def build_label(mat_path, H, W):
    # (H,W,NS) -> khop anh slice-first (NS,H,W) + xoay/dao da hieu chinh
    lab = np.transpose(rasterize_mat(mat_path, H, W), (2, 0, 1))   # (NS,H,W)
    lab = np.rot90(lab, LAB_ROT_K, axes=(1, 2))
    if LAB_FLIP: lab = lab[:, ::-1, :]
    if LAB_REV:  lab = lab[::-1]
    return np.ascontiguousarray(lab.astype(np.uint8))


## 3) Inspect 1 ca (kiểm med/lat tự nhất quán)


In [ ]:
sample = mat_paths[0]
info = parse_case(sample)
print("case:", info["case_id"], "| barcode:", info["barcode"])
vol = rasterize_mat(sample, IMG_DEFAULT_HW, IMG_DEFAULT_HW)
inv = {v: k for k, v in UNION_LABEL_NAMES.items()}
print(f"{'lab':>3} {'class':24s} {'voxels':>9}  slice_range")
for lab in sorted(set(STRUCTS.values())):
    zs = np.where((vol == lab).any((0, 1)))[0]
    rng = f"{zs.min()}..{zs.max()} (n={len(zs)})" if len(zs) else "-"
    print(f"{lab:3d} {inv.get(lab,'?'):24s} {int((vol==lab).sum()):9d}  {rng}")


## 4) QC overlay 1 ca (kiểm nhãn nằm đúng sụn trên ảnh DESS)


In [ ]:
import nibabel as nib
import matplotlib.pyplot as plt

def find_dess_image(info):
    for cand in [
        DESS_DIR / f"{info['barcode']}.nii.gz",
        DESS_DIR / f"{info['subject']}_{info['visit']}.nii.gz",
        DESS_DIR / f"{info['stem']}.nii.gz",
    ]:
        if cand.exists():
            return cand
    return None

if not LABEL_ONLY:
    imgp = find_dess_image(info)
    assert imgp is not None, f"Khong thay anh DESS cho barcode {info['barcode']} trong {DESS_DIR}"
    im  = nib.load(str(imgp)); img = np.asanyarray(im.dataobj).astype(float)   # (NS,H,W)
    lab = build_label(sample, img.shape[1], img.shape[2])
    print("img", img.shape, "| lab", lab.shape)
    assert lab.shape == img.shape, "shape lech -> kiem tra lai transform"
    z = int(np.argmax([(lab[i] > 0).sum() for i in range(lab.shape[0])]))
    imgn = (img - img.min()) / (np.ptp(img) + 1e-6)
    plt.figure(figsize=(6, 6))
    plt.imshow(imgn[z].T, cmap="gray", origin="lower")
    mm = np.ma.masked_where(lab[z] == 0, lab[z])
    plt.imshow(mm.T, cmap="nipy_spectral", alpha=0.5, origin="lower", vmin=1, vmax=8)
    plt.title(f"{info['case_id']} slice {z}"); plt.axis("off"); plt.show()
else:
    print("LABEL_ONLY=True -> bo qua QC ghep anh.")


## 5) Convert TOÀN BỘ 176 ca

Duyệt theo **ảnh DESS đã có** (khớp barcode → `.mat`). Ghi `imagesTr/CASE_0000.nii.gz` + `labelsTr/CASE.nii.gz` cùng affine. Có skip-existing + `FORCE`.


In [ ]:
from tqdm.auto import tqdm

FORCE = False

# barcode -> (.mat, case_id)
mat_by_bc = {}
for mp in mat_paths:
    ci = parse_case(mp)
    mat_by_bc[ci["barcode"]] = (mp, ci["case_id"])

def default_affine():
    return np.diag(list(DEFAULT_SPACING) + [1.0]).astype(float)

ok, skip, fail = [], [], []

if LABEL_ONLY:
    for mp in tqdm(mat_paths):
        ci = parse_case(mp); cid = ci["case_id"]
        lbl_out = LABELS_TR / f"{cid}.nii.gz"
        if not FORCE and lbl_out.exists(): skip.append(cid); ok.append(cid); continue
        try:
            lab = build_label(mp, IMG_DEFAULT_HW, IMG_DEFAULT_HW)
            nib.save(nib.Nifti1Image(lab, default_affine()), str(lbl_out)); ok.append(cid)
        except Exception as e:
            fail.append((cid, str(e))); print("LOI", cid, e)
else:
    for imgf in tqdm(sorted(DESS_DIR.glob("*.nii.gz"))):
        bc = imgf.name.replace(".nii.gz", "")
        if bc not in mat_by_bc: fail.append((bc, "khong co .mat")); continue
        mp, cid = mat_by_bc[bc]
        img_out = IMAGES_TR / f"{cid}_0000.nii.gz"
        lbl_out = LABELS_TR / f"{cid}.nii.gz"
        if not FORCE and img_out.exists() and lbl_out.exists(): skip.append(cid); ok.append(cid); continue
        try:
            im  = nib.load(str(imgf)); img = np.asanyarray(im.dataobj)
            lab = build_label(mp, img.shape[1], img.shape[2])
            if lab.shape != img.shape:
                raise ValueError(f"shape nhan {lab.shape} != anh {img.shape}")
            shutil.copy(str(imgf), str(img_out))
            nib.save(nib.Nifti1Image(lab, im.affine), str(lbl_out))
            ok.append(cid)
        except Exception as e:
            fail.append((bc, str(e))); print("LOI", bc, e)

print(f"\nXong. OK={len(ok)} (moi={len(ok)-len(skip)}, skip={len(skip)}) FAIL={len(fail)}")
for c, e in fail[:20]: print("   ", c, e)


## 6) Ghi `dataset.json`


In [ ]:
import json
labels = {k: v for k, v in sorted(UNION_LABEL_NAMES.items(), key=lambda kv: kv[1])
          if v == 0 or v in set(STRUCTS.values())}
dataset = {
    "channel_names": {"0": CHANNEL_NAME},
    "labels": labels,
    "numTraining": len(ok),
    "file_ending": ".nii.gz",
    "overwrite_image_reader_writer": "SimpleITKIO",
    "description": "OAI-iMorphics rasterized (transpose+K1+REV); nhan UNION khong bone.",
}
with open(OUT_ROOT / "dataset.json", "w") as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)
print(json.dumps(dataset, indent=2, ensure_ascii=False))


## 7) QC lại vài ca ngẫu nhiên từ dataset đã lưu


In [ ]:
import random, matplotlib.colors as mcolors
from matplotlib.patches import Patch

NAMES = {2:"femoral_cart",4:"med_tib_cart",5:"lat_tib_cart",6:"med_men",7:"lat_men",8:"patellar"}
COL   = {2:"#1f77b4",4:"#2ca02c",5:"#bcbd22",6:"#d62728",7:"#ff7f0e",8:"#17becf"}
cases = sorted(p.name.replace(".nii.gz","") for p in LABELS_TR.glob("*.nii.gz"))
print("so ca:", len(cases))
for cid in random.sample(cases, min(3, len(cases))):
    img = np.asanyarray(nib.load(str(IMAGES_TR / f"{cid}_0000.nii.gz")).dataobj).astype(float)
    lab = np.asanyarray(nib.load(str(LABELS_TR / f"{cid}.nii.gz")).dataobj)
    imgn = (img - img.min()) / (np.ptp(img) + 1e-6)
    present = [int(l) for l in np.unique(lab) if l]
    zs = sorted(int(z) for z in np.argsort([(lab[i] > 0).sum() for i in range(lab.shape[0])])[::-1][:3])
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    for a, z in zip(ax, zs):
        a.imshow(imgn[z].T, cmap="gray", origin="lower")
        rgba = np.zeros(lab[z].shape + (4,))
        for l in present:
            m = lab[z] == l
            if m.any(): rgba[m, :3] = mcolors.to_rgb(COL[l]); rgba[m, 3] = 0.55
        a.imshow(np.transpose(rgba, (1, 0, 2)), origin="lower"); a.set_title(f"{cid} z{z}"); a.axis("off")
    fig.legend(handles=[Patch(color=COL[l], label=f"{l} {NAMES[l]}") for l in present],
               loc="lower center", ncol=len(present), fontsize=8)
    plt.tight_layout(rect=[0, 0.05, 1, 1]); plt.show()
